In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.cluster import DBSCAN
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

## Caricamento dei files

In [ ]:
# Imposta il percorso della cartella contenente i file pkl
path = '/content/drive/MyDrive/cold_heads_monitoring/Dataset'
arrays = []
data_list = []

# Itera su tutti i file pkl nella cartella e caricali in una lista
for filename in os.listdir(path):
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:
            # Carica l'array NumPy dal file e seleziona solo il secondo canale
            arr = np.asarray(pickle.load(f))[:, 1:2, :]
            arrays.append(arr)
            data_list.append(os.path.splitext(filename)[0])

print(data_list)

In [ ]:
all_data = arrays[0]
#arrays = [np.array(arrays[6]), np.array(arrays[4]), np.array(arrays[2]), np.array(arrays[1]), np.array(arrays[5]), np.array(arrays[0])]

## Bilanciamento del dataset

In [ ]:
def calculate_features(window):
    # Calcola la media del segnale nella finestra
    mean = np.mean(window)

    # Calcola la deviazione standard del segnale nella finestra
    std_dev = np.std(window)

    # Calcola il massimo valore nella finestra
    max_val = np.max(window)

    # Calcola il minimo valore nella finestra
    min_val = np.min(window)

    # Calcola la differenza tra il massimo e il minimo
    peak_to_peak = max_val - min_val

    return [mean, std_dev, max_val, min_val, peak_to_peak]

In [ ]:
def normalize_data(data):
    # Calcola la media e la deviazione standard lungo ciascuna feature (colonna)
    mean = np.mean(data, axis=0)
    std_dev = np.std(data, axis=0)

    # Applica la standardizzazione (z-score normalization)
    normalized_data = (data - mean) / std_dev

    return normalized_data

In [ ]:
pca = PCA(n_components=2)

def preprocessing(signal):
    # Lunghezza della finestra e sovrapposizione
    window_size = 30
    overlap = 2

    # Calcola tutte le finestre scorrevoli
    windows = []

    n = len(signal)
    indices = np.arange(0, n - window_size + 1, overlap)
    windows.extend(signal[i:i+window_size] for i in indices)

    # Calcola le features per tutte le finestre
    features = np.array([calculate_features(window) for window in windows])

    return features


In [ ]:
dataset = preprocessing(all_data)[150]

In [ ]:
print(dataset)

## Training DBSCAN

#Inference

In [ ]:
# nuovo_campione = il tuo nuovo campione
# data = i tuoi dati di addestramento
# clustering_labels = le etichette di clustering assegnate ai dati di addestramento

def distance(nuovo_campione, data, clustering_labels, metric='chebyshev'):
    # Calcola il centroide di ciascun cluster
    cluster_centroids = []
    for label in np.unique(clustering_labels):
        cluster_points = data[clustering_labels == label]
        centroid = np.mean(cluster_points, axis=0)
        cluster_centroids.append(centroid)

    cluster_centroids = np.array(cluster_centroids)

    # Calcola la distanza tra il nuovo campione e i centroidi dei cluster
    distanze = pairwise_distances(nuovo_campione.reshape(1, -1), cluster_centroids, metric=metric)

    # Identifica il cluster più vicino
    cluster_più_vicino = np.argmin(distanze)
    return cluster_più_vicino

In [ ]:
# Carica il modello
with open('dbscan_model.pkl', 'rb') as model_file:
    loaded_dbscan = pickle.load(model_file)

In [ ]:
data= np.expand_dims(dataset,0)
print(data.shape)
# Ora puoi utilizzare loaded_dbscan per fare previsioni o ottenere etichette
clustering_labels = loaded_dbscan.labels_

new_labels = loaded_dbscan.fit_predict(data)
print(new_labels)